In [7]:
# T2.3 – Mapping Units of Measurement to DBRepo
# Frost Day Prediction in Vienna
# Owner: C (Anxhela)

# ─────────────────────────────────────────────
# STEP 1: Install and import the DBRepo library
# ─────────────────────────────────────────────
!pip install dbrepo --quiet

from dbrepo.RestClient import RestClient

# ─────────────────────────────────────────────
# STEP 2: Connect to DBRepo
# ─────────────────────────────────────────────
client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="DataStewGroup4",
    password="fakjyq-tastU6-pykcep"
)

print("Logged in as:", client.whoami())

# ─────────────────────────────────────────────
# STEP 3: Get database and table IDs
# ─────────────────────────────────────────────
DATABASE_ID = "a16a980a-3489-492b-adcf-74c70a07248b"

tables = client.get_tables(database_id=DATABASE_ID)
for t in tables:
    print(f"Table: {t.name} | ID: {t.id}")

# ─────────────────────────────────────────────
# STEP 4: Get column IDs for each table
# ─────────────────────────────────────────────
for t in tables:
    table = client.get_table(database_id=DATABASE_ID, table_id=t.id)
    print(f"\nTable: {table.name}")
    for col in table.columns:
        print(f"  Column: {col.name} | ID: {col.id}")

DataStewGroup4
Logged in as: DataStewGroup4
Table: daily_observation | ID: 21e39deb-1db9-4d34-bc04-45fa5cef72a0
Table: station | ID: 9c57f7ff-99b6-454b-8d51-cff340ecf934

Table: daily_observation
  Column: station_id | ID: df86f2ef-257b-4ce9-927a-0e85f7d30680
  Column: obs_date | ID: a016d12d-099e-488e-acd3-17d8aec2968a
  Column: temp_mean_c | ID: e94be6a3-66a0-42f4-a6ce-dff6919c286b
  Column: temp_max_c | ID: da983751-c4c5-4185-9067-fb244ca4f73e
  Column: temp_min_c | ID: 6cc7bb1d-204d-47a4-af90-76b9acca9427
  Column: precipitation_mm | ID: bd1e8916-f395-4af6-8adc-4bc65e7cee8e
  Column: sunshine_h | ID: f7ba0d31-79e9-4815-bd68-08c430c70896
  Column: humidity_pct | ID: 4ac18e04-24e7-4afe-9778-285772f3c9fa
  Column: visibility_m | ID: d7147d0d-b7af-441d-85e9-9051bf416786

Table: station
  Column: station_id | ID: a8491880-16d3-4cb4-a543-3074dba236e2
  Column: station_code | ID: 5cedc878-88cb-429e-9820-df46a82022f1
  Column: name | ID: 17bd9138-a9bd-462d-8082-f5346694ccd7
  Column: latit

In [8]:
# ─────────────────────────────────────────────
# STEP 5: Define unit mappings
# ─────────────────────────────────────────────

# SI Digital Framework URIs
SI_DEGREE      = "https://si-digital-framework.org/SI/units/degree"
SI_METRE       = "https://si-digital-framework.org/SI/units/metre"
SI_CELSIUS     = "https://si-digital-framework.org/SI/units/degreeCelsius"
SI_HOUR        = "https://si-digital-framework.org/SI/units/hour"
QUDT_PERCENT   = "http://qudt.org/vocab/unit/PERCENT"  # fallback: not in SI

# Note: millimetre uses SI metre URI (milli prefix on metre)
SI_MILLIMETRE  = "https://si-digital-framework.org/SI/units/metre"

# ─────────────────────────────────────────────
# STEP 6: Apply unit mappings
# Replace TABLE_ID and COLUMN_ID values from Step 4 output
# ─────────────────────────────────────────────

# --- station table ---
STATION_TABLE_ID = "9c57f7ff-99b6-454b-8d51-cff340ecf934"

unit_mappings_station = {
    "latitude":   SI_DEGREE,
    "longitude":  SI_DEGREE,
    "altitude_m": SI_METRE,
}

station_table = client.get_table(database_id=DATABASE_ID, table_id=STATION_TABLE_ID)
for col in station_table.columns:
    if col.name in unit_mappings_station:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=STATION_TABLE_ID,
            column_id=col.id,
            unit_uri=unit_mappings_station[col.name]
        )
        print(f"✅ station.{col.name} → {unit_mappings_station[col.name]}")

# --- daily_observation table ---
OBS_TABLE_ID = "21e39deb-1db9-4d34-bc04-45fa5cef72a0"

unit_mappings_obs = {
    "temp_mean_c":      SI_CELSIUS,
    "temp_max_c":       SI_CELSIUS,
    "temp_min_c":       SI_CELSIUS,
    "precipitation_mm": SI_MILLIMETRE,
    "sunshine_h":       SI_HOUR,
    "humidity_pct":     QUDT_PERCENT,
    "visibility_m":     SI_METRE,
}

obs_table = client.get_table(database_id=DATABASE_ID, table_id=OBS_TABLE_ID)
for col in obs_table.columns:
    if col.name in unit_mappings_obs:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=OBS_TABLE_ID,
            column_id=col.id,
            unit_uri=unit_mappings_obs[col.name]
        )
        print(f"✅ daily_observation.{col.name} → {unit_mappings_obs[col.name]}")

print("\n✅ All unit mappings applied successfully!")

✅ station.latitude → https://si-digital-framework.org/SI/units/degree
✅ station.longitude → https://si-digital-framework.org/SI/units/degree
✅ station.altitude_m → https://si-digital-framework.org/SI/units/metre
✅ daily_observation.temp_mean_c → https://si-digital-framework.org/SI/units/degreeCelsius
✅ daily_observation.temp_max_c → https://si-digital-framework.org/SI/units/degreeCelsius
✅ daily_observation.temp_min_c → https://si-digital-framework.org/SI/units/degreeCelsius
✅ daily_observation.precipitation_mm → https://si-digital-framework.org/SI/units/metre
✅ daily_observation.sunshine_h → https://si-digital-framework.org/SI/units/hour
✅ daily_observation.humidity_pct → http://qudt.org/vocab/unit/PERCENT
✅ daily_observation.visibility_m → https://si-digital-framework.org/SI/units/metre

✅ All unit mappings applied successfully!


In [9]:
# Verify unit mappings were saved
obs_table = client.get_table(database_id=DATABASE_ID, table_id=OBS_TABLE_ID)
for col in obs_table.columns:
    print(f"{col.name} | unit_uri: {col.unit_uri}")

station_table = client.get_table(database_id=DATABASE_ID, table_id=STATION_TABLE_ID)
for col in station_table.columns:
    print(f"{col.name} | unit_uri: {col.unit_uri}")

station_id | unit_uri: None
obs_date | unit_uri: None
temp_mean_c | unit_uri: https://si-digital-framework.org/SI/units/degreeCelsius
temp_max_c | unit_uri: https://si-digital-framework.org/SI/units/degreeCelsius
temp_min_c | unit_uri: https://si-digital-framework.org/SI/units/degreeCelsius
precipitation_mm | unit_uri: https://si-digital-framework.org/SI/units/metre
sunshine_h | unit_uri: https://si-digital-framework.org/SI/units/hour
humidity_pct | unit_uri: http://qudt.org/vocab/unit/PERCENT
visibility_m | unit_uri: https://si-digital-framework.org/SI/units/metre
station_id | unit_uri: None
station_code | unit_uri: None
name | unit_uri: None
latitude | unit_uri: https://si-digital-framework.org/SI/units/degree
longitude | unit_uri: https://si-digital-framework.org/SI/units/degree
altitude_m | unit_uri: https://si-digital-framework.org/SI/units/metre
